In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pendulibrary.integrate import integrate_state
from pendulibrary.continuation import adaptive_cont
from pendulibrary.targeter import dc_underconstrained
from pendulibrary.common_targetters import single_fixed
from pendulibrary.interpolate import interp_hermite
from scipy.interpolate import CubicSpline

int_tol = 1e-14

# Read in the file

In [ ]:
fname = "DD_short_period"

data = np.load(f"../database/{fname}.npz")
x0s = data["x0s"]
periods = data["periods"]
tangents = data["tangents"]
eigs = data["eigs"]
Lr, Mr = data["params"]

# Plot the old $f(\nu(s))$ just so we know what to expect

In [ ]:
# Plot the bifurcations of the old file

nu_coarse = np.sum(eigs, axis=1).real - 2

ind_fix = np.argmin(np.std(x0s), axis=0)
Xs_coarse = np.column_stack((x0s, periods))
Xs_coarse = np.delete(Xs_coarse, ind_fix, 1)
Ss_coarse = np.append(0, np.cumsum(np.linalg.norm(np.diff(Xs_coarse, axis=0), axis=1)))

fig, axs = plt.subplots()
plt.plot([Ss_coarse[0], Ss_coarse[-1]], [0, 0], "--k")
axs.plot(Ss_coarse, nu_coarse - 2, "-m", label=r"$\lambda=1$")
axs.plot(Ss_coarse, nu_coarse + 2, "-g", label=r"Double")
axs.plot(Ss_coarse, nu_coarse + 1, "-b", label=r"Truple")
axs.plot(Ss_coarse, nu_coarse, "-c", label=r"Quadrouple")
axs.set(ylim=(-3, 3))
axs.legend()
plt.show()

# Refine the grid by a lot so it's much smoother

In [ ]:
targ = single_fixed(ind_fix, x0s[ind_fix][0], 2, Lr, Mr, int_tol)
func = targ.g_dg_stm

ds = 1e-3
Sfin = 4.0
Ss = np.arange(0, Sfin + ds, ds)
N = Ss.shape[0]
Xs_guess = interp_hermite(Ss_coarse, Xs_coarse.T, tangents.T, Ss)[1].T

Xs = np.empty_like(Xs_guess)
eigs = np.empty((N, 4), np.complex128)
tangs = np.empty((N, 4))
for ii in tqdm(range(N)):
    X, DF, stm = dc_underconstrained(Xs_guess[ii], func, 1e-12, max_iter=25)
    Xs[ii] = X.copy()
    eigs[ii] = np.linalg.eigvals(stm).copy()
    tangs[ii] = np.linalg.svd(DF).Vh[-1].copy()

nu = (np.sum(eigs, axis=1) - 2).real
Ss = np.append(0, np.cumsum(np.linalg.norm(np.diff(Xs, axis=0), axis=1)))
f1 = nu - 2
f2 = nu + 2
f3 = nu + 1
f4 = nu
root_fs = [f1, f2, f3, f4]

# Plot the new suspected bifurcation points

In [ ]:
splines = [CubicSpline(Ss, f) for f in root_fs]
allroots = [spline.roots(extrapolate=False) for spline in splines]
cols = ["m", "g", "b", "c"]
labels = ["Bifurcation", "Double", "Triple", "Quadrouple"]

fig, ax = plt.subplots()
plt.plot([Ss[0], Ss[-1]], [0, 0], "--k")
for ii in range(4):
    ax.plot(Ss, root_fs[ii], f"-{cols[ii]}", label=labels[ii])
    ax.plot(allroots[ii], 0 * allroots[ii], f".{cols[ii]}", ms=10)
    for jj in range(len(allroots[ii])):
        ax.text(allroots[ii][jj], 0.5, f"P{ii+1}:{jj}")
ax.set(ylim=(-3, 3))
ax.legend()
plt.show()

# Find the exact new bifurcation points

In [ ]:
tangs_bif = [[], [], [], []]
Xs_bif = [[], [], [], []]
eigs_bif = [[], [], [], []]
Xs_guess = [interp_hermite(Ss, Xs.T, tangs.T, rts)[1].T for rts in allroots]
for ii in range(4):
    for jj in range(len(Xs_guess[ii])):
        X, df, stm = dc_underconstrained(Xs_guess[ii][jj], func, 1e-12, max_iter=25)
        X[-1] *= 1 + ii
        X, df, stm = dc_underconstrained(X, func, 1e-12, max_iter=25)
        Xs_bif[ii].append(X.copy())
        tangs_bif[ii].append(np.linalg.svd(DF).Vh[-2])
        eigs_bif[ii].append(np.linalg.eigvals(stm))

# Continue a whole family of em

In [ ]:
cont_kwargs = dict(
    s0=1e-3, s_min=1e-5, S=1.5, tol=1e-9, max_iter=25, target_iter=10, rate=1.15
)


bif_type = 3
j_bif = 0
Xs_post, eig_vals_post, (DFs_post, tangents_post, stms_post) = adaptive_cont(
    Xs_bif[bif_type][j_bif],
    func,
    tangs_bif[bif_type][j_bif],
    **cont_kwargs,
    exact_tangent=True,
)

In [ ]:
x0s_new = np.array([targ.get_x0(X) for X in Xs_post])
periods_new = np.array([targ.get_period(X) for X in Xs_post])
np.savez(
    "../database/DD_short_period_P3a",
    x0s=x0s_new,
    periods=periods_new,
    eigs=eig_vals_post,
    tangents=tangents_post,
    params=np.array([Lr, Mr]),
)

# Optional plotters

In [ ]:
from pendulibrary.plotters import plot_timeline_grid, make_gif

Ss_new = np.append(0, np.cumsum(np.linalg.norm(np.diff(Xs_post, axis=0), axis=1)))

N = 6
for j in range(N):
    s = j * Ss_new[-1] / N
    iii = np.argmin(np.abs(Ss_new - s))
    ts, xs, fs = integrate_state(x0s_new[iii], periods_new[iii], Lr, Mr, 1e-14)
    fig = plot_timeline_grid(xs, ts, fs, Lr)
    fig.savefig(f"fig_{j}.png")
    plt.close(fig)

In [ ]:
ts, xs, fs = integrate_state(x0s_new[-1], periods_new[-1], Lr, Mr, 1e-14)
make_gif(xs, ts, fs, Lr, "test3", 150, (4, 4))